# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riteshy1526/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents the daily performance of a single content page for one client.

For this assignment, I use the **March 2026 (month=2026-03)** partition because it is a mid-panel month and avoids using the final month as the outcome window.

The data contains observed daily search performance and engagement signals that can support refresh-priority decisions.

One row represents the daily performance of a single content page.

For this assignment, I use the **March 2026 (month=2026-03)** partition because it is a mid-panel month and avoids using the final month as the outcome window.

This data is used to understand historical content performance and support refresh-priority decisions.

In [27]:
import duckdb

con = duckdb.connect()

path = r"C:\Users\AJIT KUMAR YADAV\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet"

query = f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date,
COUNT(DISTINCT content_hash_id) AS unique_pages
FROM read_parquet('{path}')
"""

con.execute(query).df()

,total_rows,start_date,end_date,unique_pages
0,9841378,2026-03-01,2026-03-31,331437


## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

These features are available before making the content refresh decision.

### Label / Proxy
Refresh Opportunity Score (proxy for future improvement after refreshing content).

### Context
- report_date
- client_hash_id
- content_hash_id
- month

These fields identify the content page and reporting period.

### Excluded
Future performance after refresh is excluded because it is not available at the decision time and would introduce data leakage.

### Features
- impressions
- clicks
- ctr
- average_position
- content_age_days

These features are available before making the refresh decision.

### Label / Proxy
Refresh Opportunity Score (proxy for future improvement after content refresh).

### Context
- content_id
- client_id
- date

These identify the content and provide context but are not prediction targets.

### Excluded
Future outcome variables are excluded because they would introduce data leakage.

In [28]:
df = con.execute(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

The following queries verify the data contract.

- One row represents one content page on one reporting date.
- The selected partition covers March 2026.
- Data availability is verified using warehouse availability flags.6

The following queries verify the data contract:

1. One row represents one content page for one day.
2. The selected month is March 2026.
3. Required fields are available and contain usable values.

In [29]:
con.execute(f"""
SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT content_hash_id) AS unique_pages
FROM read_parquet('{path}')
""").df()

,total_rows,unique_pages
0,9841378,331437


In [30]:
con.execute(f"""
SELECT
MIN(report_date) AS first_day,
MAX(report_date) AS last_day
FROM read_parquet('{path}')
""").df()

,first_day,last_day
0,2026-03-01,2026-03-31


In [31]:
con.execute(f"""
SELECT
SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM read_parquet('{path}')
""").df()

,gsc_available_rows,ga4_available_rows
0,3611061.0,413966.0


## 4. Data limits

This dataset contains historical observational data only.

It cannot directly measure the causal impact of refreshing content.

Some GA4 metrics may be unavailable for clients without GA4 integration.

Future outcomes are unknown at the decision point, so the model should be used for decision support rather than guaranteed prediction.

In [32]:
print("Known limitation documented.")

Known limitation documented.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.